In [1]:
import pandas as pd

In [9]:
df = pd.read_csv('affairs.csv', )
df = df.iloc[:,1:]
df

,rate_marriage,age,yrs_married,children,religious,educ,occupation,occupation_husb,affairs
0,3.0,32.0,9.0,3.0,3.0,17.0,2.0,5.0,0.111111
1,3.0,27.0,13.0,3.0,1.0,14.0,3.0,4.0,3.230769
2,4.0,22.0,2.5,0.0,1.0,16.0,3.0,5.0,1.400000
3,4.0,37.0,16.5,4.0,3.0,16.0,5.0,5.0,0.727273
4,5.0,27.0,9.0,1.0,1.0,14.0,3.0,4.0,4.666666
...,...,...,...,...,...,...,...,...,...
6361,5.0,32.0,13.0,2.0,3.0,17.0,4.0,3.0,0.000000
6362,4.0,32.0,13.0,1.0,1.0,16.0,5.0,5.0,0.000000
6363,5.0,22.0,2.5,0.0,2.0,14.0,3.0,1.0,0.000000
6364,5.0,32.0,6.0,1.0,3.0,14.0,3.0,4.0,0.000000


In [10]:
df.head()

,rate_marriage,age,yrs_married,children,religious,educ,occupation,occupation_husb,affairs
0,3.0,32.0,9.0,3.0,3.0,17.0,2.0,5.0,0.111111
1,3.0,27.0,13.0,3.0,1.0,14.0,3.0,4.0,3.230769
2,4.0,22.0,2.5,0.0,1.0,16.0,3.0,5.0,1.400000
3,4.0,37.0,16.5,4.0,3.0,16.0,5.0,5.0,0.727273
4,5.0,27.0,9.0,1.0,1.0,14.0,3.0,4.0,4.666666


In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# 0) 타깃 생성
# =========================
df['affair_flag'] = (df['affairs'] > 0).astype(int)

X = df.drop(columns=['affairs', 'affair_flag'])
y = df['affair_flag']

# =========================
# 1) 홀드아웃: Train / Test
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# =========================
# 2) 파이프라인 & K-Fold 설정
# =========================
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',   # 불균형 대응
        solver='lbfgs'
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'acc': 'accuracy',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall',
    'roc_auc': 'roc_auc'
}

# =========================
# 3) K-Fold Validation (Train 내부)
# =========================
cv_results = cross_validate(
    pipe, X_train, y_train,
    cv=cv, scoring=scoring, return_train_score=False, n_jobs=-1
)

def mean_std(x):
    return f"{np.mean(x):.4f} ± {np.std(x):.4f}"

print("=== 5-Fold CV (Train 내부 Validation) ===")
for k, v in cv_results.items():
    if k.startswith('test_'):
        print(f"{k.replace('test_', '').upper():>9}: {mean_std(v)}")

# =========================
# 4) 전체 Train으로 재학습 후 Test 평가
# =========================
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print("\n=== Test 결과 (Hold-out) ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


=== 5-Fold CV (Train 내부 Validation) ===
      ACC: 0.6876 ± 0.0090
       F1: 0.5713 ± 0.0087
PRECISION: 0.5126 ± 0.0110
   RECALL: 0.6456 ± 0.0122
  ROC_AUC: 0.7463 ± 0.0136

=== Test 결과 (Hold-out) ===
Accuracy : 0.6766
ROC AUC  : 0.7246

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.70      0.74       863
           1       0.50      0.64      0.56       411

    accuracy                           0.68      1274
   macro avg       0.65      0.67      0.65      1274
weighted avg       0.70      0.68      0.68      1274

Confusion Matrix:
[[600 263]
 [149 262]]


In [13]:
df = pd.read_csv('affairs.csv', )
df = df.iloc[:,1:]
df

,rate_marriage,age,yrs_married,children,religious,educ,occupation,occupation_husb,affairs
0,3.0,32.0,9.0,3.0,3.0,17.0,2.0,5.0,0.111111
1,3.0,27.0,13.0,3.0,1.0,14.0,3.0,4.0,3.230769
2,4.0,22.0,2.5,0.0,1.0,16.0,3.0,5.0,1.400000
3,4.0,37.0,16.5,4.0,3.0,16.0,5.0,5.0,0.727273
4,5.0,27.0,9.0,1.0,1.0,14.0,3.0,4.0,4.666666
...,...,...,...,...,...,...,...,...,...
6361,5.0,32.0,13.0,2.0,3.0,17.0,4.0,3.0,0.000000
6362,4.0,32.0,13.0,1.0,1.0,16.0,5.0,5.0,0.000000
6363,5.0,22.0,2.5,0.0,2.0,14.0,3.0,1.0,0.000000
6364,5.0,32.0,6.0,1.0,3.0,14.0,3.0,4.0,0.000000


In [15]:
# ============================================
# Affairs 이진분류: Train/Val/Test + K-Fold + 튜닝 + 임계값 최적화 (수정본)
# ============================================
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV, cross_val_predict
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

# --------------------------------------------
# 0) 가정: df(DataFrame)가 이미 존재
#    컬럼: ['rate_marriage','age','yrs_married','children','religious',
#           'educ','occupation','occupation_husb','affairs']
# --------------------------------------------

# 1) 타깃 생성
df = df.copy()
df['affair_flag'] = (df['affairs'] > 0).astype(int)

# 2) Train / Test 분리 (홀드아웃)
X_full = df.drop(columns=['affairs', 'affair_flag'])
y_full = df['affair_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# 3) 피처 엔지니어링 함수 (신규 컬럼 2개 생성)
ENGINEERED_COLS = ['yrs_per_age', 'rate_x_yrs']

def add_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    X['yrs_per_age'] = (X['yrs_married'] / X['age']).replace([np.inf, -np.inf], np.nan).fillna(0)
    X['rate_x_yrs'] = X['rate_marriage'] * X['yrs_married']
    return X

# ✅ 핵심 수정: feature_names_out을 callable로 지정 (입력컬럼 + 신규컬럼 반환)
def feat_names_out(transformer, input_features):
    return list(input_features) + ENGINEERED_COLS

feat_eng = FunctionTransformer(
    add_features,
    validate=False,
    feature_names_out=feat_names_out
)

# 4) 컬럼 타입 정의
categorical_cols = ['occupation', 'occupation_husb']
# ✅ 신규 엔지니어드 컬럼을 numeric 목록에 포함 (파이프라인에서 feat_eng 이후에 prep이 참조)
numeric_cols = [c for c in X_train.columns if c not in categorical_cols] + ENGINEERED_COLS

# 5) 전처리 파이프라인
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ],
    remainder='drop'
)

# 6) 모델 정의
logreg = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    solver='lbfgs'
)

rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# 7) 파이프라인
pipe_logreg = Pipeline(steps=[
    ('feat', feat_eng),
    ('prep', preprocessor),
    ('clf', logreg)
])

pipe_rf = Pipeline(steps=[
    ('feat', feat_eng),
    ('prep', preprocessor),
    ('clf', rf)
])

# 8) K-Fold & 스코어링
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'acc': 'accuracy',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall',
    'roc_auc': 'roc_auc'
}

# 9) 기본 K-Fold 성능 비교
print("=== 5-Fold CV: Logistic Regression ===")
cv_log = cross_validate(pipe_logreg, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
for k, v in cv_log.items():
    if k.startswith('test_'):
        print(f"{k.replace('test_', '').upper():>9}: {np.mean(v):.4f} ± {np.std(v):.4f}")

print("\n=== 5-Fold CV: Random Forest ===")
cv_rf = cross_validate(pipe_rf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
for k, v in cv_rf.items():
    if k.startswith('test_'):
        print(f"{k.replace('test_', '').upper():>9}: {np.mean(v):.4f} ± {np.std(v):.4f}")

# 10) GridSearchCV (ROC AUC 기준)
param_grid_logreg = {
    'clf__C': [0.1, 1.0, 5.0, 10.0],
    'clf__penalty': ['l2']
}
param_grid_rf = {
    'clf__n_estimators': [300, 500],
    'clf__max_depth': [None, 8, 12],
    'clf__min_samples_leaf': [1, 3, 5]
}

print("\n=== GridSearchCV: Logistic Regression (ROC AUC) ===")
gs_log = GridSearchCV(
    estimator=pipe_logreg, param_grid=param_grid_logreg,
    scoring='roc_auc', cv=cv, n_jobs=-1, verbose=0
)
gs_log.fit(X_train, y_train)
print("Best ROC AUC:", gs_log.best_score_)
print("Best Params:", gs_log.best_params_)

print("\n=== GridSearchCV: Random Forest (ROC AUC) ===")
gs_rf = GridSearchCV(
    estimator=pipe_rf, param_grid=param_grid_rf,
    scoring='roc_auc', cv=cv, n_jobs=-1, verbose=0
)
gs_rf.fit(X_train, y_train)
print("Best ROC AUC:", gs_rf.best_score_)
print("Best Params:", gs_rf.best_params_)

# 11) 베스트 모델 선택
best_model = gs_rf if gs_rf.best_score_ >= gs_log.best_score_ else gs_log
print("\n=== Selected Best Model:", type(best_model.best_estimator_['clf']).__name__, "===")

# 12) 임계값 최적화 (OOF 기반 F1 최대화)
oof_proba = cross_val_predict(
    best_model.best_estimator_, X_train, y_train,
    cv=cv, method='predict_proba', n_jobs=-1
)[:, 1]

thresholds = np.linspace(0.2, 0.8, 61)
f1s = []
for th in thresholds:
    preds = (oof_proba >= th).astype(int)
    f1s.append(f1_score(y_train, preds))
best_th = thresholds[int(np.argmax(f1s))]
print(f"\nBest threshold by OOF F1: {best_th:.3f}  (F1={np.max(f1s):.4f})")

# 13) 전체 Train 재학습 후 Test 평가
best_est = best_model.best_estimator_
best_est.fit(X_train, y_train)

test_proba = best_est.predict_proba(X_test)[:, 1]
test_pred_default = (test_proba >= 0.5).astype(int)
test_pred_bestth = (test_proba >= best_th).astype(int)

def evaluate(y_true, y_pred, y_proba, tag=""):
    print(f"\n=== Test Evaluation ({tag}) ===")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1       : {f1_score(y_true, y_pred):.4f}")
    print(f"ROC AUC  : {roc_auc_score(y_true, y_proba):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

evaluate(y_test, test_pred_default, test_proba, tag="threshold=0.50")
evaluate(y_test, test_pred_bestth, test_proba, tag=f"threshold={best_th:.2f}")


=== 5-Fold CV: Logistic Regression ===
      ACC: 0.6932 ± 0.0133
       F1: 0.5880 ± 0.0186
PRECISION: 0.5186 ± 0.0156
   RECALL: 0.6791 ± 0.0241
  ROC_AUC: 0.7574 ± 0.0147

=== 5-Fold CV: Random Forest ===
      ACC: 0.6911 ± 0.0034
       F1: 0.4934 ± 0.0129
PRECISION: 0.5237 ± 0.0063
   RECALL: 0.4671 ± 0.0230
  ROC_AUC: 0.7044 ± 0.0032

=== GridSearchCV: Logistic Regression (ROC AUC) ===
Best ROC AUC: 0.7574317183835244
Best Params: {'clf__C': 5.0, 'clf__penalty': 'l2'}

=== GridSearchCV: Random Forest (ROC AUC) ===
Best ROC AUC: 0.7572944164918061
Best Params: {'clf__max_depth': 8, 'clf__min_samples_leaf': 5, 'clf__n_estimators': 300}

=== Selected Best Model: LogisticRegression ===

Best threshold by OOF F1: 0.420  (F1=0.5968)

=== Test Evaluation (threshold=0.50) ===
Accuracy : 0.6860
Precision: 0.5102
Recall   : 0.6715
F1       : 0.5798
ROC AUC  : 0.7333

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.69      0.75  

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

def evaluate_with_threshold(model, X, y, threshold=0.5):
    """
    학습된 모델(model)과 데이터(X, y)를 받아서
    임계값(threshold)에 따라 성능 지표 출력
    """
    # 예측 확률
    y_proba = model.predict_proba(X)[:, 1]
    # 임계값 적용
    y_pred = (y_proba >= threshold).astype(int)
    
    # 성능 지표
    print(f"\n=== Evaluation (threshold={threshold:.2f}) ===")
    print(f"Accuracy : {accuracy_score(y, y_pred):.4f}")
    print(f"Precision: {precision_score(y, y_pred):.4f}")
    print(f"Recall   : {recall_score(y, y_pred):.4f}")
    print(f"F1 Score : {f1_score(y, y_pred):.4f}")
    print(f"ROC AUC  : {roc_auc_score(y, y_proba):.4f}")
    print("\nClassification Report:")
    print(classification_report(y, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y, y_pred))
    
    return y_pred, y_proba

# ---------------------------------
# 사용 예시
# ---------------------------------
# best_est 는 이미 학습된 최종 모델이라고 가정

# 기본 임계값 0.5로 평가
evaluate_with_threshold(best_est, X_test, y_test, threshold=0.50)

# 최적화된 임계값 0.42로 평가
evaluate_with_threshold(best_est, X_test, y_test, threshold=0.42)

# 최적화된 임계값 0.42로 평가
evaluate_with_threshold(best_est, X_test, y_test, threshold=0.36)

# 최적화된 임계값 0.42로 평가
evaluate_with_threshold(best_est, X_test, y_test, threshold=0.28)


=== Evaluation (threshold=0.50) ===
Accuracy : 0.6860
Precision: 0.5102
Recall   : 0.6715
F1 Score : 0.5798
ROC AUC  : 0.7333

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.69      0.75       863
           1       0.51      0.67      0.58       411

    accuracy                           0.69      1274
   macro avg       0.66      0.68      0.66      1274
weighted avg       0.72      0.69      0.69      1274

Confusion Matrix:
[[598 265]
 [135 276]]

=== Evaluation (threshold=0.42) ===
Accuracy : 0.6358
Precision: 0.4603
Recall   : 0.7470
F1 Score : 0.5696
ROC AUC  : 0.7333

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.58      0.68       863
           1       0.46      0.75      0.57       411

    accuracy                           0.64      1274
   macro avg       0.64      0.66      0.63      1274
weighted avg       0.71      0.64      0.65      1274

Con

(array([0, 1, 1, ..., 0, 1, 0]),
 array([0.1559585 , 0.31725187, 0.68631003, ..., 0.19368145, 0.3818978 ,
        0.13250725]))

In [18]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, accuracy_score

def optimize_threshold_by_profit(
    y_true,
    y_proba,
    revenue_tp=1.0,   # TP(실제 양성 잡음) 1건당 수익
    cost_fp=1.0,      # FP(실제 음성인데 양성으로 의심) 1건당 추가 조사 비용
    cost_fn=0.0,      # FN(놓침)의 기회비용(있으면 +로 넣기, 기본 0)
    benefit_tn=0.0    # TN(정상으로 잘 걸러짐) 1건당 절감/혜택(있으면 +로 넣기)
):
    """
    수익 함수: Profit = TP*revenue_tp - FP*cost_fp - FN*cost_fn + TN*benefit_tn
    반환: best_threshold, summary(dict), per_cutoff(dict of arrays)
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_proba).astype(float)
    n = len(y_true)
    assert n == len(y_score), "y_true와 y_proba 길이가 달라요."

    # 점수 내림차순 정렬
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    s_sorted = y_score[order]

    # 누적 TP/FP 계산 (k개를 양성으로 판단한다고 가정: 상위 k개)
    cum_pos = np.cumsum(y_sorted)           # k개 중 양성 개수 = TP_k
    idx = np.arange(1, n+1)
    tp = cum_pos
    fp = idx - tp

    # 전체 양성/음성
    P = y_true.sum()
    N = n - P

    # 나머지 FN/TN
    fn = P - tp
    tn = N - fp

    # 수익 계산
    profit = tp*revenue_tp - fp*cost_fp - fn*cost_fn + tn*benefit_tn

    # 최적 컷오프 k 찾기 (k=0도 고려: 아무도 양성으로 안 잡음)
    # k=0 처리 위해 0 지점 추가
    tp0 = 0
    fp0 = 0
    fn0 = P - tp0
    tn0 = N - fp0
    profit0 = tp0*revenue_tp - fp0*cost_fp - fn0*cost_fn + tn0*benefit_tn

    profit_all = np.concatenate([[profit0], profit])
    tp_all = np.concatenate([[tp0], tp])
    fp_all = np.concatenate([[fp0], fp])
    fn_all = np.concatenate([[fn0], fn])
    tn_all = np.concatenate([[tn0], tn])

    best_k = int(np.argmax(profit_all))  # 0..n
    best_profit = float(profit_all[best_k])

    # 임계값 계산: k개의 상위 샘플을 양성으로 보면
    # threshold는 s_sorted[k-1]과 s_sorted[k] 사이의 중간값 (경계 처리 포함)
    if best_k == 0:
        # 모두 음성 예측 → 최대 점수보다 큰 임계값
        best_threshold = min(1.0, float(s_sorted[0]) + 1e-12)
    elif best_k == n:
        # 모두 양성 예측 → 최소 점수보다 작은 임계값
        best_threshold = max(0.0, float(s_sorted[-1]) - 1e-12)
    else:
        left = float(s_sorted[best_k-1])
        right = float(s_sorted[best_k])
        best_threshold = (left + right) / 2.0

    summary = {
        "best_threshold": best_threshold,
        "best_profit": best_profit,
        "TP": int(tp_all[best_k]),
        "FP": int(fp_all[best_k]),
        "FN": int(fn_all[best_k]),
        "TN": int(tn_all[best_k]),
        "P": int(P),
        "N": int(N),
    }

    per_cutoff = {
        "k": np.arange(0, n+1),
        "profit": profit_all,
        "TP": tp_all,
        "FP": fp_all,
        "FN": fn_all,
        "TN": tn_all,
    }

    return best_threshold, summary, per_cutoff


def evaluate_at_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "confusion_matrix": cm
    }


In [19]:
# 가정: best_est 는 학습 완료된 최종 파이프라인/모델
y_proba_test = best_est.predict_proba(X_test)[:, 1]
y_true_test  = y_test.values

# 1) 기본 시나리오: TP=+1, FP=-0.3 (조사비용), FN=-1.5 (놓치면 큰 손실), TN=+0
best_th, info, curve = optimize_threshold_by_profit(
    y_true_test, y_proba_test,
    revenue_tp=1.0,   # TP 수익
    cost_fp=0.3,      # FP 비용
    cost_fn=1.5,      # FN 비용
    benefit_tn=0.0    # TN 혜택
)

print(">>> Best threshold (by profit):", round(best_th, 4))
print(">>> Profit summary:", info)

# 2) 해당 임계값에서 성능 지표
report = evaluate_at_threshold(y_true_test, y_proba_test, best_th)
print("\n>>> Metrics @ best threshold")
for k, v in report.items():
    if k != "confusion_matrix":
        print(f"{k}: {v}")
print("Confusion Matrix:\n", report["confusion_matrix"])


>>> Best threshold (by profit): 0.132
>>> Profit summary: {'best_threshold': 0.13199684935351658, 'best_profit': 173.9, 'TP': 410, 'FP': 782, 'FN': 1, 'TN': 81, 'P': 411, 'N': 863}

>>> Metrics @ best threshold
threshold: 0.13199684935351658
accuracy: 0.38540031397174257
precision: 0.34395973154362414
recall: 0.9975669099756691
f1: 0.511540860885839
roc_auc: 0.7332580569675748
Confusion Matrix:
 [[ 81 782]
 [  1 410]]


In [20]:
# ===========================================
# 특정 입력값으로 예측 확률 / 결과 확인
# ===========================================

import pandas as pd

def predict_single_case(best_est, input_data, threshold=None):
    """
    best_est : 학습 완료된 파이프라인 모델
    input_data : dict 형태 (컬럼명: 값)
    threshold : None이면 기본 0.5, 값 지정 시 해당 임계값 적용
    """
    # DataFrame 변환
    df_input = pd.DataFrame([input_data])

    # 예측 확률
    proba = best_est.predict_proba(df_input)[:, 1][0]

    # 예측 결과
    if threshold is None:
        threshold = 0.5
    pred = int(proba >= threshold)

    return {
        "threshold": threshold,
        "predicted_affair_flag": pred,   # 1이면 affairs 있음
        "probability": proba
    }

# ==========================
# 예시 입력 (컬럼명은 학습 데이터와 동일하게)
# ==========================
sample_case = {
    'rate_marriage': 4.0,
    'age': 35.0,
    'yrs_married': 10.0,
    'children': 2.0,
    'religious': 2.0,
    'educ': 16.0,
    'occupation': 3.0,
    'occupation_husb': 4.0
}

# 1) 기본 threshold=0.5
result_default = predict_single_case(best_est, sample_case)
print("기본 기준(0.5):", result_default)

# 2) 최적 threshold(best_th) 사용
result_best = predict_single_case(best_est, sample_case, threshold=best_th)
print(f"최적 기준({best_th:.2f}):", result_best)


기본 기준(0.5): {'threshold': 0.5, 'predicted_affair_flag': 1, 'probability': np.float64(0.5556004931283116)}
최적 기준(0.13): {'threshold': 0.13199684935351658, 'predicted_affair_flag': 1, 'probability': np.float64(0.5556004931283116)}


# 저장 불러오는 코드

In [21]:
import joblib

# 저장 파일 경로
model_path = "best_model_with_threshold.pkl"

# 저장할 객체에 모델과 threshold 같이 묶기
model_bundle = {
    "model": best_est,
    "best_threshold": best_th
}

# 저장
joblib.dump(model_bundle, model_path)
print(f"모델과 임계값 저장 완료: {model_path}")


모델과 임계값 저장 완료: best_model_with_threshold.pkl


In [22]:
# 불러오기
loaded_bundle = joblib.load(model_path)

# 모델과 threshold 꺼내기
loaded_model = loaded_bundle["model"]
loaded_threshold = loaded_bundle["best_threshold"]

print("불러온 threshold:", loaded_threshold)


불러온 threshold: 0.13199684935351658


In [23]:
# 예시 입력
sample_case = {
    'rate_marriage': 4.0,
    'age': 35.0,
    'yrs_married': 10.0,
    'children': 2.0,
    'religious': 2.0,
    'educ': 16.0,
    'occupation': 3.0,
    'occupation_husb': 4.0
}

import pandas as pd

df_input = pd.DataFrame([sample_case])

# 예측 확률
prob = loaded_model.predict_proba(df_input)[:, 1][0]

# threshold 적용
pred = int(prob >= loaded_threshold)

print(f"확률: {prob:.4f}, 예측: {pred} (1=affair 있음)")


확률: 0.5556, 예측: 1 (1=affair 있음)
